In [0]:
%run ../config_setup

In [0]:
# Databricks Notebook: sim_01_rds_cdc (parameterized)
# Schedule: Every 2 min (test) / Every 30 min (prod)
# Requires: sim_00_seed must have run once

import uuid, random, datetime as dt, json
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

ROOT = LANDING
SEED = f"{ROOT}/_seed"

# ============================================================
# 1. SAFETY — seed must exist
# ============================================================
def seed_exists():
    try:
        return spark.read.json(f"{SEED}/machines").count() > 0
    except Exception:
        return False

if not seed_exists():
    dbutils.notebook.exit(
        "❌ SEED MISSING — run sim_00_seed notebook first (one-time)"
    )
print("✅ Seed found")

# ============================================================
# 2. Load seed as lists (FK references)
# ============================================================
machines = [r.asDict() for r in spark.read.json(f"{SEED}/machines").collect()]
models   = [r.asDict() for r in spark.read.json(f"{SEED}/machine_models").collect()]
owners   = [r.asDict() for r in spark.read.json(f"{SEED}/owners").collect()]
operators= [r.asDict() for r in spark.read.json(f"{SEED}/operators").collect()]
sites    = [r.asDict() for r in spark.read.json(f"{SEED}/sites").collect()]
parts    = [r.asDict() for r in spark.read.json(f"{SEED}/materials").collect()]

machine_ids  = [m["machine_id"] for m in machines]
model_ids    = [m["model_id"] for m in models]
owner_ids    = [o["owner_id"] for o in owners]
operator_ids = [o["operator_id"] for o in operators]
site_ids     = [s["site_id"] for s in sites]
part_ids     = [p["material_id"] for p in parts]

print(f"   FK refs: {len(machine_ids)} machines, {len(site_ids)} sites, "
      f"{len(operator_ids)} operators, {len(part_ids)} parts")

# ============================================================
# 3. Helper — rounding without Spark clash
# ============================================================
def rnd(x, n=6):
    return float(f"{x:.{n}f}")

# ============================================================
# 4. Run metadata
# ============================================================
run_ts = dt.datetime.utcnow()
run_id = f"{run_ts:%Y%m%dT%H%M%S}_{uuid.uuid4().hex[:6]}"
print(f"\nRun ID: {run_id}")

# ============================================================
# 5. CDC event helpers
# ============================================================
def emit(table, op, before, after):
    return {
        "op": {"insert":"C","update":"U","delete":"D"}[op],
        "_change_type": op,
        "_lsn": random.randint(1000000, 9999999),
        "_commit_ts": run_ts.isoformat() + "Z",
        "table": table,
        "before": json.dumps(before) if before else None,
        "after": json.dumps(after) if after else None,
    }

from pyspark.sql.types import StructType, StructField, StringType, LongType

def write_events(table, events):
    if not events: return
    
    schema = StructType([
        StructField("op", StringType(), True),
        StructField("_change_type", StringType(), True),
        StructField("_lsn", LongType(), True),
        StructField("_commit_ts", StringType(), True),
        StructField("table", StringType(), True),
        StructField("before", StringType(), True),
        StructField("after", StringType(), True)
    ])
    
    df = spark.createDataFrame(events, schema=schema)
    df.write.mode("append").json(f"{ROOT}/rds/{table}/{run_id}.json")
    print(f"  ✓ rds.{table}: {len(events)} events")

# ============================================================
# 6. Generate CDC for 11 tables
# ============================================================

# ---- 1. machine_master ----
# 👉 FIX: insert ab SEED machines (MCH-0001..0050) emit karta hai.
# Pehle f"MCH-{randint(100,999)}" (3-digit) generate karta tha jo
# telemetry ke machines se kabhi match nahi karte the — isi wajah se
# dim_machine me sirf 2 alien machines the aur facts me machine_sk NULL the.
events = []
for _ in range(random.randint(3, 6)):
    op = random.choices(["insert","update","delete"], weights=[0.5,0.35,0.15])[0]
    if op == "insert":
        m = random.choice(machines).copy()
        m["status"] = random.choice(["active","idle","maintenance"])
        events.append(emit("machine_master", "insert", None, m))
    elif op == "update":
        m = random.choice(machines).copy()
        m["status"] = random.choice(["active","idle","maintenance"])
        events.append(emit("machine_master", "update", None, m))
    else:
        events.append(emit("machine_master", "delete", random.choice(machines), None))
write_events("machine_master", events)

# ---- 2. machine_model ----
write_events("machine_model", [emit("machine_model", "update", None, random.choice(models))
                               for _ in range(random.randint(1,3))])

# ---- 3. owner ----
write_events("owner", [emit("owner", "update", None, random.choice(owners))
                       for _ in range(random.randint(1,2))])

# ---- 4. operator ----
write_events("operator", [emit("operator", "insert", None, {
    "operator_id": f"OPR-{random.randint(100,999)}",
    "operator_name": f"NewOp_{random.randint(1,99)}",
    "license_no": f"LIC{random.randint(10000,99999)}",
    "owner_id": random.choice(owner_ids),
}) for _ in range(random.randint(1,3))])

# ---- 5. operator_assignment ----
write_events("operator_assignment", [emit("operator_assignment", "insert", None, {
    "assignment_id": f"ASN-{uuid.uuid4().hex[:8]}",
    "machine_id": random.choice(machine_ids),
    "operator_id": random.choice(operator_ids),
    "shift_start": run_ts.isoformat()+"Z",
    "shift_end": (run_ts + dt.timedelta(hours=8)).isoformat()+"Z",
}) for _ in range(random.randint(2,5))])

# ---- 6. site ----
write_events("site", [emit("site", "insert", None, {
    "site_id": f"SITE-{random.randint(20,99)}",
    "site_name": f"NewSite_{random.randint(1,99)}",
    "country_code": "IN",
    "geo_lat": rnd(random.uniform(-30,55), 6),
    "geo_lon": rnd(random.uniform(-120,145), 6),
}) for _ in range(random.randint(0,2))])

# ---- 7. work_order ----
write_events("work_order", [emit("work_order", "insert", None, {
    "work_order_id": f"WO-{random.randint(10000,99999)}",
    "machine_id": random.choice(machine_ids),
    "operator_id": random.choice(operator_ids),
    "site_id": random.choice(site_ids),
    "start_ts": run_ts.isoformat()+"Z",
    "end_ts": (run_ts + dt.timedelta(hours=4)).isoformat()+"Z",
    "status": random.choice(["open","in_progress","closed"]),
}) for _ in range(random.randint(3,6))])

# ---- 8. maintenance_record ----
write_events("maintenance_record", [emit("maintenance_record", "insert", None, {
    "maint_id": f"MNT-{random.randint(10000,99999)}",
    "machine_id": random.choice(machine_ids),
    "maint_type": random.choice(["preventive","corrective","emergency"]),
    "cost": rnd(random.uniform(100,5000), 2),
    "completed_ts": run_ts.isoformat()+"Z",
}) for _ in range(random.randint(1,3))])

# ---- 9. parts_inventory ----
write_events("parts_inventory", [emit("parts_inventory", "update", None, {
    "part_id": random.choice(part_ids),
    "stock_qty": random.randint(0,500),
    "reorder_level": random.randint(10,100),
}) for _ in range(random.randint(1,2))])

# ---- 10. maintenance_parts_used ----
write_events("maintenance_parts_used", [emit("maintenance_parts_used", "insert", None, {
    "maint_id": f"MNT-{random.randint(10000,99999)}",
    "part_id": random.choice(part_ids),
    "qty_used": random.randint(1,10),
    "unit_cost": rnd(random.uniform(10,500), 2),
}) for _ in range(random.randint(1,3))])

# ---- 11. work_order_status_history ----
write_events("work_order_status_history", [emit("work_order_status_history", "insert", None, {
    "history_id": f"HST-{uuid.uuid4().hex[:8]}",
    "work_order_id": f"WO-{random.randint(10000,99999)}",
    "old_status": random.choice(["open","in_progress"]),
    "new_status": random.choice(["in_progress","closed"]),
    "changed_ts": run_ts.isoformat()+"Z",
}) for _ in range(random.randint(2,4))])

print(f"\n✅ RDS run complete: {run_id}")